In [1]:
import json 
import pandas as pd
import numpy 
import matplotlib.pyplot as plt 
import seaborn as sns

ModuleNotFoundError: No module named 'pandas'

In [ ]:
# input_path = 'arxiv_sample_50k.json'
# data = []
# with open(input_path, 'r') as f:
#     for line in f:
#         data.append(json.loads(line))
df = pd.read_json('arxiv_sample_50k.json', lines = True)


In [ ]:
df.head()

,id,authors,title,categories,abstract
0,2303.06576,"E. Grohs, A. B. Balantekin",Implications on Cosmology from Dirac Neutrino ...,hep-ph astro-ph.CO,The mechanism for generating neutrino masses...
1,1101.3145,"H.Q. Yuan, J. Chen, J. Singleton, S. Akutagawa...",Large upper critical field in non-centrosymmet...,cond-mat.supr-con,We determine the upper critical field $\mu_0...
2,1506.05616,"Jaya Kumar A., Buddhapriya Chakrabarti and Yas...",Equilibrium of fluid membranes with tangent-pl...,cond-mat.soft,Fluid membranes endowed with tangent-plane o...
3,2408.06921,"Chao Yu, Qi Xu, and Jun Zhang",Recent advances in InGaAs/InP single-photon de...,physics.ins-det quant-ph,Single-photon detectors (SPDs) are widely us...
4,2508.17313,"Alexander Alban, Fu-Hsuan Ho, Justin Ko",The Free Energy of an Enriched Continuous Rand...,math.PR math.AP,We revisit the proof of the limiting free ener...


In [ ]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   id          50000 non-null  str  
 1   authors     50000 non-null  str  
 2   title       50000 non-null  str  
 3   categories  50000 non-null  str  
 4   abstract    50000 non-null  str  
dtypes: str(5)
memory usage: 1.9 MB
None


In [ ]:
print(sum(df.duplicated()))

0


In [ ]:
df['paper_info'] = df['title'] + ' ' + df['abstract']

In [ ]:
columns = ['title' , 'abstract']
df = df.drop(columns, axis = 1)


In [ ]:
df.head()

,id,authors,categories,paper_info
0,2303.06576,"E. Grohs, A. B. Balantekin",hep-ph astro-ph.CO,Implications on Cosmology from Dirac Neutrino ...
1,1101.3145,"H.Q. Yuan, J. Chen, J. Singleton, S. Akutagawa...",cond-mat.supr-con,Large upper critical field in non-centrosymmet...
2,1506.05616,"Jaya Kumar A., Buddhapriya Chakrabarti and Yas...",cond-mat.soft,Equilibrium of fluid membranes with tangent-pl...
3,2408.06921,"Chao Yu, Qi Xu, and Jun Zhang",physics.ins-det quant-ph,Recent advances in InGaAs/InP single-photon de...
4,2508.17313,"Alexander Alban, Fu-Hsuan Ho, Justin Ko",math.PR math.AP,The Free Energy of an Enriched Continuous Rand...


In [ ]:
len(df['categories'].unique())

7007

In [ ]:
df['info'] = df['paper_info'] + ' ' + 'Categories' + ' ' + df['categories']

In [ ]:
df = df.drop(columns=['paper_info', 'authors', 'categories'])

In [ ]:
df.head()

,id,info
0,2303.06576,Implications on Cosmology from Dirac Neutrino ...
1,1101.3145,Large upper critical field in non-centrosymmet...
2,1506.05616,Equilibrium of fluid membranes with tangent-pl...
3,2408.06921,Recent advances in InGaAs/InP single-photon de...
4,2508.17313,The Free Energy of an Enriched Continuous Rand...


In [ ]:
df['info'] = df['info'].str.replace('\n', ' ')

df['info'] = df['info'].str.strip()

df['info'] = df['info'].str.lower()

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

# 1. Load the Tokenizer and the Model
model_name = "sentence-transformers/all-MiniLM-L6-v2"

print("Downloading/Loading tokenizer and model...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

/opt/anaconda3/envs/nlp_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]


Downloading/Loading tokenizer and model...


/opt/anaconda3/envs/nlp_env/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [ ]:
# 1. Extract the text from your DataFrame
texts = df['info'].tolist()

# 2. Tokenize the text (convert words to numbers the model understands)
print("Tokenizing text...")
encoded_input = tokenizer(texts, padding=True, truncation=True, return_tensors='pt')

# 3. Pass the tokens through the model to get the raw outputs
print("Passing text through the model (this might take a moment)...")
with torch.no_grad(): # This disables training mode to save memory and speed things up
    model_output = model(**encoded_input)

# 4. Define and apply mean pooling to get a single vector per sentence
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0] # First element contains all token embeddings
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

print("Applying mean pooling...")
sentence_embeddings = mean_pooling(model_output, encoded_input['attention_mask'])

# 5. Convert the PyTorch tensor to a standard Python list and add it to your DataFrame
df['embeddings'] = sentence_embeddings.tolist()

print("Success! Here is a look at your updated DataFrame:")
print(df.head())

Tokenizing text...
Passing text through the model (this might take a moment)...


: 